### Sample Code to train and Deploy Time Series Forecast Model

##### Install Snowpark
##### Install prophet model on your local instance

In [ ]:
# !pip install snowflake-snowpark-python[pandas]==0.10.0
# !pip install prophet
# !pip install statsmodels
# !pip install joblib

In [1]:
from snowflake.snowpark import version
print(f"snowflake snowpark version is: {version.VERSION}")

snowflake snowpark version is: (1, 0, 0)


##### Import snowpark libraries and other required libraries

In [2]:
# Snowpark
import snowflake.snowpark 
from snowflake.snowpark.functions import sproc
import snowflake.snowpark.types as T
from snowflake.snowpark.session import Session
from snowflake.snowpark import version as v
import json
from config import snowflake_conn_prop_local as snowflake_udf_conn_prop

import pandas as pd
import numpy as np
import datetime
import io

##### Create Snowflake connection

In [3]:
session = Session.builder.configs(snowflake_udf_conn_prop).create()
print(session.sql('select current_account(), current_warehouse(), current_database(), current_schema()').collect())
session.sql("use role accountadmin").collect()
session.sql("use database mlflow_poc_ktn").collect()
session.sql("use schema public").collect()
session.add_packages('snowflake-snowpark-python', 'pandas', 'numpy', 'statsmodels', 'matplotlib','joblib','prophet')

[Row(CURRENT_ACCOUNT()='ANA95816', CURRENT_WAREHOUSE()='APP_WH', CURRENT_DATABASE()='MLFLOW_POC_KTN', CURRENT_SCHEMA()='PUBLIC')]


The version of package prophet in the local environment is 1.1.2, which does not fit the criteria for the requirement prophet. Your UDF might not work when the package version is different between the server and your local environment


##### Create internal stage to save model files and dependencies if not already created

In [4]:
# session.sql('create or replace stage models').collect()

##### Define and register stored proc for prophet model

In [7]:
import pandas as pd
from prophet import Prophet
import os
from joblib import dump

@sproc(name="prophet_forecast_occ", 
       is_permanent=True, 
       stage_location="@models", 
       replace=True, 
       packages=["snowflake-snowpark-python", "prophet","joblib"])
def prophet_forecast_occ(session: snowflake.snowpark.Session, 
                         raw_table: str, 
                         out_table_name: str,
                         litigation_type: str,
                         case_category: str,
                         origination_court: str,
                         case_type: str,
                         mod_name: str) -> T.Variant:
    import pandas as pd
    from prophet import Prophet
    import os
    from joblib import dump
    
    def fix_date_cols(df, tz='UTC'):
        cols = df.select_dtypes(include=['datetime64[ns]']).columns
        for col in cols:
            df[col] = df[col].dt.tz_localize(tz)
    
    litigation_type_ = "'" + litigation_type + "'"
    case_category_ = "'" + case_category + "'"
    origination_court_ = "'" + origination_court + "'"
    case_type_ = "'" + case_type + "'"
    
    ### Get the source table
    qr1 = '''
        SELECT *
        FROM ''' + raw_table + ''' 
        WHERE litigation_type = ''' + litigation_type_ + ''' 
            and case_category = ''' + case_category_ + ''' 
            and originating_court = ''' + origination_court_ + ''' 
            and case_type = ''' + case_type_
    
    dat = session.sql(qr1).to_pandas()
    df = dat[['FILEDDATE', 'COUNT']]
    df.columns = ['ds', 'y']
    
    ### Define and the Model
    model = Prophet()
    model.fit(df)
    
    ### Create future data
    future = model.make_future_dataframe(periods=60, include_history=True, freq = 'MS')
    # future.tail()
    
    ### Predict using the model above
    forecast = model.predict(future)
    
    ### Using Pandas dataframe assemble the output
    # forecast[['ds', 'yhat', 'yhat_lower', 'yhat_upper']].tail()
    df['forecast'] = forecast['yhat']
    df.columns = ['Date', 'y', 'forecast']
    
    df1 = pd.concat([forecast, df], axis=1)[["ds", "y", "yhat"]]
    df1.columns = ['Date', 'Actual', 'Predicted']
    
    ### Convert date to datetime so SF reads it correctly
    fix_date_cols(df1)
    
    # ### Write Predictions to snowflake table
    # snowpark_df = session.write_pandas(df1, out_table_name, auto_create_table=True, overwrite=True)
    
    ### Derive model metrics squared error(se), mean square error(mse) and root mean square error(rmse) on prediction
    se = np.square(forecast.loc[:, 'yhat'] - df['y'])
    mse = np.mean(se)
    rmse = np.sqrt(mse)
    

    # ### Get Model metrics in a dataframe for the model
    df_metrics =  pd.DataFrame({'litigation_type':[litigation_type], 
                                'case_category':[case_category], 
                                'originating_court':[origination_court], 
                                'case_type':[case_type]
                                ,'model':[mod_name],
                                'rmse':[rmse]})

    
    ### Write to Model metrics to snowflake table
    session.create_dataframe(df_metrics).write.mode("append").save_as_table(mod_name + "_metrics")
    
    # Upload trained model to internal stage
    model_file = os.path.join('/tmp', mod_name + '.joblib')
    dump(model, model_file)
    session.file.put(model_file, "@models",overwrite=True)
    
    # return root mean square for the model
    
    return rmse

/opt/anaconda3/envs/snowpark_012/lib/python3.8/site-packages/tqdm/auto.py:22: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Importing plotly failed. Interactive plots will not work.
The version of package prophet in the local environment is 1.1.2, which does not fit the criteria for the requirement prophet. Your UDF might not work when the package version is different between the server and your local environment


##### Call the Stored Proc to train the timeseries forecast prophet model and predict. Make sure the stored proc works as expected for one-time run

In [8]:
mod_name = 'prophet'
raw_table = 'dummy_data'
out_table_name = 'dummy_data_5y_' + mod_name
litigation_type = 'Litigation Type 1'
case_category = 'Case Category 1'
origination_court = 'JC 1'
case_type = 'Case Type 1'

prophet_forecast_occ(raw_table,
                     out_table_name,
                     litigation_type,
                     case_category,
                     origination_court,
                     case_type,
                     mod_name
)

'16.775806829822585'

### You can build multiple models using one registered stored proc and run them parallel using snowflake tasks from 1 stored proc. Code below

In [98]:
@sproc(name="create_multi_task", 
       is_permanent=True, 
       stage_location="@models", 
       replace=True, 
       packages=["snowflake-snowpark-python"])
def create_multi_task(session: snowflake.snowpark.Session, 
                         raw_table: str, 
                         out_table_name: str,
                         model_stored_proc: str,
                         parent_task_name: str,
                         child_task_name: str,
                         mod_name: str) -> T.Variant:
    qry = f"select distinct litigation_type, case_category, originating_court, case_type from {raw_table}"
    df1 = session.sql(qry).to_pandas()

    out_table_name = out_table_name + mod_name

    create_parent_task = f"create or replace task {parent_task_name} schedule = '100 MINUTE' warehouse = COMPUTE_WH allow_overlapping_execution = false as select current_timestamp;"

    #Suspend the parent immediately as there is a schedule
    suspend_parent_task = f"alter task {parent_task_name} suspend;"

    # print (create_parent_task)
    session.sql(create_parent_task).collect()
    session.sql(suspend_parent_task).collect()

    for i in range(0,len(df1)):
        litigation_type = df1.iloc[i][0]
        case_category = df1.iloc[i][1]
        origination_court = df1.iloc[i][2]
        case_type = df1.iloc[i][3]

        task_prefix = f"CREATE or replace TASK T_{i}_{child_task_name} warehouse = MULTI_TASK_WRHS allow_overlapping_execution = false AFTER {parent_task_name} AS "

        sp = (f"call {model_stored_proc}{raw_table,out_table_name, litigation_type, case_category,origination_court,case_type,mod_name};")

        create_child_task = task_prefix+sp
        # print (create_child_task)

        resume_child_task = f"alter task T_{i}_{child_task_name} resume;"
        # print (resume_child_task)

        session.sql(create_child_task).collect()
        session.sql(resume_child_task).collect()
    return 'task creation complete....please validate'

### Set the input parameters for the task creation. Remember this will not execute the tasks as the parent task is suspended during the creation.

In [10]:
mod_name = 'prophet'
raw_table = 'dummy_data'
out_table_name = 'dummy_data_5y_'
model_stored_proc = 'prophet_forecast_occ'
parent_task_name = 'PARENT_PROPHET'
child_task_name = 'CHILD_PROPHET'
model_name = 'PROPHET'

### Execute the task Creation on the database and validate the task graph in snowflake

In [105]:
create_multi_task(raw_table,
                 out_table_name,
                 model_stored_proc,
                 parent_task_name,
                 child_task_name,
                 mod_name)

'"task creation complete....please validate"'

### Truncate the metrics table for each model

In [8]:
session.sql("truncate table prophet_metrics").collect()

[Row(status='Statement executed successfully.')]

### Now you that you have validated, please go ahead and execute the parent task. The parent task will continue to be in suspended state. If you want it to be schedule please resume the task.

In [11]:
from datetime import datetime
now = datetime.now()
print("Parent Task Start Time :- ", now)

execute_parent = f"execute task {parent_task_name};"
print (execute_parent)
session.sql(execute_parent).collect()

Parent Task Start Time :-  2023-07-26 15:45:08.863856
execute task PARENT_PROPHET;


[Row(status='Task PARENT_PROPHET is scheduled to run immediately.')]

### Before the task history is reflect with all the executions, if you are aware of how many tasks are executing, you can validate their scores for each combination in the metrics we capture in snowflake table.

In [15]:
session.sql("select * from prophet_metrics").to_pandas()

,litigation_type,case_category,originating_court,case_type,model,rmse
0,Litigation Type 1,Case Category 1,JC 4,Case Type 1,prophet,123.786113
1,Litigation Type 1,Case Category 1,JC 1,Case Type 8,prophet,85.069754
2,Litigation Type 1,Case Category 3,JC 6,Case Type 2,prophet,67.304808
3,Litigation Type 1,Case Category 3,JC 6,Case Type 7,prophet,76.531204
4,Litigation Type 1,Case Category 1,JC 6,Case Type 10,prophet,80.397814
...,...,...,...,...,...,...
75,Litigation Type 1,Case Category 1,JC 6,Case Type 6,prophet,85.842424
76,Litigation Type 1,Case Category 3,JC 4,Case Type 7,prophet,79.536894
77,Litigation Type 1,Case Category 1,JC 1,Case Type 6,prophet,86.114273
78,Litigation Type 1,Case Category 3,JC 2,Case Type 16,prophet,73.560955


### After the tasks have completed, run the below query to check the stats and performance. The below list of 80 tasks were executed with M standard warehouse and 10 multicluster after trying different combinations. You may need to change based on your SLA if the tasks are failing

### The stats will take roughly 10-15 mins to reflect on task_history but the tasks would have executed/executing. Refer to the time recorded above to make sure you are checking the stats associated to your run.

In [22]:
task_series = 'PROPHET'
check_task_completion = f'''with abc as (select *, DATEDIFF(SECOND, SCHEDULED_TIME, COMPLETED_TIME) AS DURATION
from 
snowflake.account_usage.task_history
where database_name = 'MLFLOW_POC_KTN' and name like '%{task_series}%') 
select name, state, scheduled_time, completed_time , DURATION
from abc where 
run_id = (select max(run_id) from abc)
order by 5 DESC
'''
session.sql (check_task_completion).to_pandas()

,NAME,STATE,SCHEDULED_TIME,COMPLETED_TIME,DURATION
0,T_46_CHILD_PROPHET,SUCCEEDED,2023-07-26 15:45:17.183000-07:00,2023-07-26 15:46:23.334000-07:00,66
1,T_60_CHILD_PROPHET,SUCCEEDED,2023-07-26 15:45:17.183000-07:00,2023-07-26 15:46:23.374000-07:00,66
2,T_49_CHILD_PROPHET,SUCCEEDED,2023-07-26 15:45:17.183000-07:00,2023-07-26 15:46:22.394000-07:00,65
3,T_12_CHILD_PROPHET,SUCCEEDED,2023-07-26 15:45:17.183000-07:00,2023-07-26 15:46:22.334000-07:00,65
4,T_76_CHILD_PROPHET,SUCCEEDED,2023-07-26 15:45:17.183000-07:00,2023-07-26 15:46:22.374000-07:00,65
...,...,...,...,...,...
76,T_38_CHILD_PROPHET,SUCCEEDED,2023-07-26 15:45:17.183000-07:00,2023-07-26 15:45:35.414000-07:00,18
77,T_63_CHILD_PROPHET,SUCCEEDED,2023-07-26 15:45:17.183000-07:00,2023-07-26 15:45:35.394000-07:00,18
78,T_30_CHILD_PROPHET,SUCCEEDED,2023-07-26 15:45:17.183000-07:00,2023-07-26 15:45:35.394000-07:00,18
79,T_7_CHILD_PROPHET,SUCCEEDED,2023-07-26 15:45:17.183000-07:00,2023-07-26 15:45:35.414000-07:00,18


### Since all the tasks were executed in parallel with the same start time, the total time to execute is the same as how max duration of any task in the DAG.

In [24]:
session.close()